# 21 · Sequence log-probabilities, reward models, DPO and its variants

**Needed for:** InstructGPT-style RLHF (RL track, step R3) and DPO (step R4).

Preference learning starts from pairs: for a prompt, response $y_w$ was preferred over $y_l$.
- **Reward model + RL:** fit a scalar reward $r(x, y)$ so preferred responses score higher (Bradley-Terry), then
  optimize the policy against it with PPO (notebook 20).
- **DPO:** skip the reward model. The KL-regularized RL objective has a closed-form optimum, so the policy's own
  log-probability ratios against a frozen reference *are* an implicit reward, and you fit them directly.

Both need the same primitive, computed correctly: the log-probability of a response under a model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Sequence log-probability from logits

The logits at position $t$ predict token $t+1$, so shift by one: `logits[:, :-1]` scores `input_ids[:, 1:]`.
Sum only over **response** tokens (mask out the prompt and padding). Sum, not mean: DPO compares total
log-probabilities of whole responses. (Averaging changes the objective; that's what SimPO does on purpose.)

In [ ]:
def sequence_logprobs(logits, input_ids, response_mask):
    """logits: (B, T, V); input_ids, response_mask: (B, T), mask = 1 on response tokens.
    Returns (B,) summed log-prob of the response tokens."""
    logp = logits[:, :-1].log_softmax(-1)
    token_logp = logp.gather(-1, input_ids[:, 1:, None]).squeeze(-1)     # (B, T-1)
    return (token_logp * response_mask[:, 1:]).sum(-1)


def test_seq_logprobs(fn):
    torch.manual_seed(0)
    B, T, V = 2, 6, 11
    logits, ids = torch.randn(B, T, V), torch.randint(0, V, (B, T))
    mask = torch.tensor([[0, 0, 1, 1, 1, 0], [0, 1, 1, 1, 1, 1]]).float()   # prompt, response, padding
    ref = torch.zeros(B)
    for b in range(B):
        for t in range(1, T):
            if mask[b, t]:
                ref[b] += logits[b, t - 1].log_softmax(-1)[ids[b, t]]
    torch.testing.assert_close(fn(logits, ids, mask), ref)


test_seq_logprobs(sequence_logprobs)
print("sequence log-probs match the loop reference")

## 2. Bradley-Terry reward model

$P(y_w \succ y_l) = \sigma\big(r(y_w) - r(y_l)\big)$, so the loss is $-\log\sigma(r_w - r_l)$. Only reward
*differences* are identified: adding a constant changes nothing.

Toy setup: each response is a feature vector; a hidden true reward is linear in the features; preference labels are
sampled from the Bradley-Terry model (so they're noisy, like human labels). Fit a linear reward model.

In [ ]:
def bt_loss(r_chosen, r_rejected):
    return -F.logsigmoid(r_chosen - r_rejected).mean()


D = 8
true_w = torch.randn(D)


def preference_pairs(n):
    a, b = torch.randn(n, D), torch.randn(n, D)
    a_wins = torch.rand(n) < torch.sigmoid(a @ true_w - b @ true_w)
    chosen = torch.where(a_wins[:, None], a, b)
    rejected = torch.where(a_wins[:, None], b, a)
    return chosen, rejected


rm = nn.Linear(D, 1, bias=False)
opt = torch.optim.Adam(rm.parameters(), lr=1e-2)
for step in range(500):
    c, r = preference_pairs(256)
    loss = bt_loss(rm(c).squeeze(-1), rm(r).squeeze(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()

a, b = torch.randn(5000, D), torch.randn(5000, D)
truth = (a @ true_w) > (b @ true_w)
pred = (rm(a) > rm(b)).squeeze(-1)
cos = F.cosine_similarity(rm.weight.flatten(), true_w, dim=0).item()
print(f"reward model agrees with the true ranking on {(pred == truth).float().mean():.1%} of new pairs; "
      f"cosine(learned w, true w) = {cos:.3f}")

## 3. DPO

The KL-regularized objective $\max_\pi \mathbb{E}[r] - \beta\,\text{KL}(\pi\,\|\,\pi_{ref})$ is maximized by
$\pi^*(y) \propto \pi_{ref}(y)\,e^{r(y)/\beta}$. Rearranged, $r(y) = \beta \log\frac{\pi^*(y)}{\pi_{ref}(y)} + \text{const}$.
Plug that into Bradley-Terry and the constant cancels:

$$\mathcal{L}_{DPO} = -\log\sigma\Big(\beta\big[(\log\pi_\theta(y_w) - \log\pi_{ref}(y_w)) - (\log\pi_\theta(y_l) - \log\pi_{ref}(y_l))\big]\Big)$$

Inputs are four sequence log-probs per pair (from section 1). The reference log-probs are computed once, with no
gradient.

In [ ]:
def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    """All (B,) summed sequence log-probs. Returns (loss, implicit reward margin)."""
    logits = beta * ((pi_chosen - ref_chosen) - (pi_rejected - ref_rejected))
    return -F.logsigmoid(logits).mean(), logits.detach() / beta


def test_dpo(fn):
    pc = torch.tensor([-5.0, -3.0], requires_grad=True)
    pr = torch.tensor([-4.0, -6.0], requires_grad=True)
    rc, rr = torch.tensor([-5.0, -3.0]), torch.tensor([-4.0, -6.0])
    loss, _ = fn(pc, pr, rc, rr, beta=0.1)
    torch.testing.assert_close(loss, torch.log(torch.tensor(2.0)))   # policy == reference: loss = log 2
    loss.backward()
    assert (pc.grad < 0).all() and (pr.grad > 0).all()                # descent raises chosen, lowers rejected


test_dpo(dpo_loss)
print("DPO loss is log 2 at the reference and pushes chosen up / rejected down")

## 4. DPO recovers the analytic optimum

Ten possible responses to one prompt. The "policy" is just a logit per response. Preferences come from a hidden
reward via Bradley-Terry on pairs drawn from the reference. After training, the policy should match
$\pi_{ref}\,e^{r/\beta}$ (normalized). Smaller $\beta$ means a larger move away from the reference.

In [ ]:
K = 10
torch.manual_seed(1)
ref_logits = torch.randn(K) * 0.5
true_r = torch.randn(K)
ref_logp_all = ref_logits.log_softmax(0)

for beta in (1.0, 0.3):
    theta = nn.Parameter(ref_logits.clone())
    opt = torch.optim.Adam([theta], lr=0.05)
    for step in range(3000):
        i = torch.multinomial(ref_logp_all.exp(), 512, replacement=True)
        j = torch.multinomial(ref_logp_all.exp(), 512, replacement=True)
        keep = i != j
        i, j = i[keep], j[keep]
        i_wins = torch.rand(len(i)) < torch.sigmoid(true_r[i] - true_r[j])
        w, l = torch.where(i_wins, i, j), torch.where(i_wins, j, i)
        logp = theta.log_softmax(0)
        loss, _ = dpo_loss(logp[w], logp[l], ref_logp_all[w], ref_logp_all[l], beta=beta)
        opt.zero_grad()
        loss.backward()
        opt.step()
    learned = theta.detach().softmax(0)
    optimum = (ref_logp_all + true_r / beta).softmax(0)
    kl = (learned * (learned.log() - ref_logp_all.exp().log())).sum().item()
    print(f"beta={beta}: total variation to the analytic optimum {0.5 * (learned - optimum).abs().sum().item():.3f}   "
          f"KL(policy||ref) {kl:.3f}   P(best response) {learned[true_r.argmax()].item():.2f}")

## 5. Variants you'll meet

| Method | Change from DPO |
|---|---|
| IPO | squared loss $(\text{margin} - \frac{1}{2\beta})^2$, so it doesn't push the margin to infinity on deterministic preferences |
| SimPO | no reference model; uses **length-normalized** (mean) log-probs and a target margin $\gamma$ |
| KTO | works from single thumbs-up / thumbs-down labels instead of pairs |
| ORPO | adds an odds-ratio preference term to plain SFT; no reference model |

In [ ]:
def ipo_loss(pi_c, pi_r, ref_c, ref_r, beta=0.1):
    margin = (pi_c - ref_c) - (pi_r - ref_r)
    return ((margin - 1 / (2 * beta)) ** 2).mean()


def simpo_loss(pi_c_mean, pi_r_mean, beta=2.0, gamma=0.5):
    """Inputs are MEAN per-token log-probs of each response."""
    return -F.logsigmoid(beta * (pi_c_mean - pi_r_mean) - gamma).mean()


ipo = ipo_loss(torch.tensor([-5.0]), torch.tensor([-6.0]), torch.tensor([-5.0]), torch.tensor([-5.0]))
simpo = simpo_loss(torch.tensor([-1.0]), torch.tensor([-1.5]))
print(f"IPO loss for a margin of 1 at beta 0.1 (target 5): {ipo.item():.1f}   SimPO loss example: {simpo.item():.3f}")

## Things to be able to explain

- Why does DPO's chosen log-probability often *decrease* during training even while the margin grows? (Both go
  down, rejected faster; a known failure mode that motivates adding an SFT term or using other variants.)
- Why must the reference model be the SFT model the data was generated near? (The implicit reward is relative to it.)
- Reward hacking with an explicit reward model: why can't DPO over-optimize the reward model the same way, and what
  can still go wrong? (No separate RM to exploit, but it can over-fit to the preference data.)

## Drill

In [ ]:
def my_sequence_logprobs(logits, input_ids, response_mask):
    raise NotImplementedError


def my_dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    """Return (loss, margin)."""
    raise NotImplementedError


drill(test_seq_logprobs, my_sequence_logprobs)
drill(test_dpo, my_dpo_loss)